# Testar o extrator em páginas novas (sem viés)

Roda a comparação **justa** — extrator × detector YOLO × híbrido — contra a sua anotação humana de um lote Diamond. Para o veredito real, use um lote **desenhado do zero** (`diamond_teste`), onde não há ancoragem.

**Como usar:** anote o lote na ferramenta Diamond, escolha o `LOTE` abaixo e rode tudo (Run All).

Métricas (maior é melhor): **F1@0,5** = achou a região; **F1 c/ classe** = achou *e* classificou; **Classe (%)** = fração de palavras com a classe certa.

In [1]:
# --- Configuração ---
LOTE  = 'diamond_teste'   # lote a testar (do zero = sem viés)
PESOS = 'models/detector_diamond/kaggle_best_limpo.pt'   # pesos do detector YOLO

import os, sys, subprocess, collections
root = os.getcwd()
while not os.path.isdir(os.path.join(root, 'src')) and os.path.dirname(root) != root:
    root = os.path.dirname(root)
os.chdir(root); sys.path.insert(0, root)
print('projeto:', root)

projeto: C:\Users\pedro\dev\tcc-ri-reading-order


In [2]:
# --- 1. Confere a anotação humana ---
import src.data.regioes as seg
from src.eval.avaliar_diamond import medir_pagina, resumir, ler_jsonl
gold = {a['page_id']: a for a in ler_jsonl(f'data/diamond/{LOTE}/anotacoes.jsonl') if a.get('status') == 'concluida'}
tar  = {t['page_id']: t for t in ler_jsonl(f'data/diamond/{LOTE}/tarefas.jsonl')}
pronto = len(gold) > 0
print(f'{len(gold)} páginas concluídas em {LOTE} (de {len(tar)} tarefas)')
if not pronto:
    print('>> Anote o lote primeiro na ferramenta Diamond (servidor.py --diamond) e rode de novo.')

0 páginas concluídas em diamond_teste (de 12 tarefas)
>> Anote o lote primeiro na ferramenta Diamond (servidor.py --diamond) e rode de novo.


In [3]:
# --- 2. Roda o detector YOLO no lote (gera as caixas + classes previstas) ---
pred_file = f'resultados/deteccao/pred_{LOTE}.jsonl'
if pronto:
    subprocess.run([sys.executable, 'src/models/detector_layout.py', 'prever',
                    '--pesos', PESOS, '--lote', LOTE, '--saida', pred_file], check=True)
    yolo = {x['page_id']: x['regioes'] for x in ler_jsonl(pred_file)}
    print('predições YOLO:', len(yolo))

In [4]:
# --- 3. Compara: extrator x YOLO x híbrido (2 variantes) ---
import pandas as pd
def avaliar(gera):
    tot = collections.Counter()
    for pid, a in gold.items():
        t = tar[pid]; p = gera(t, pid)
        if p:
            tot.update(medir_pagina(t['palavras'], a['regioes'], p))
    r = resumir(tot)
    return {'F1@0,5': r['f1_j50'], 'F1 c/ classe': r['f1_j50_classe'],
            'Classe (%)': round(100 * r['classe_certa_palavras'], 1)}

if pronto:
    ext = lambda t, pid: seg.propor(t['palavras'], t['blocos_v3'], t['detector']['regioes'], regras=True)
    hib = lambda conj: (lambda t, pid: seg.aplicar_classes_rose(ext(t, pid), t['palavras'], yolo.get(pid) or [], conj))
    sistemas = {
        'Extrator (regras)':             ext,
        'Detector YOLO':                 lambda t, pid: yolo.get(pid),
        'Híbrido - todas as classes':    hib('todas'),
        'Híbrido - só assin.+cabeçalho': hib('assinatura_cabecalho'),
    }
    df = pd.DataFrame({nome: avaliar(g) for nome, g in sistemas.items()}).T
    print(f'{len(gold)} páginas · lote {LOTE}')
    display(df.style.format({'F1@0,5': '{:.3f}', 'F1 c/ classe': '{:.3f}', 'Classe (%)': '{:.1f}'})
            .background_gradient(cmap='Greens', axis=0))

### Como ler

- **F1@0,5** é a localização (achar a caixa). Extrator e híbrido têm o **mesmo** valor — o híbrido usa as caixas do extrator e só troca a classe.
- **F1 c/ classe** e **Classe (%)** dizem se a classificação melhorou. Se o híbrido superar o extrator aqui, o detector classifica melhor que as regras.
- ⚠️ **Só confie no resultado se o `LOTE` for um lote do zero** (`diamond_teste`). Em `diamond_piloto` / `diamond_treino` o gabarito foi corrigido a partir do extrator (**ancoragem**), então ele aparece inflado e a comparação é injusta.